In [ ]:
import numpy as np
from tensorflow.keras.models import load_model
from tkinter import Tk, filedialog, Label, Button, Text, END
from PIL import Image, ImageTk, ImageSequence

# ===============================
# LOAD MODEL
# ===============================
model = load_model("cnn_model.h5")

# ===============================
# CLASS LABELS
# ===============================
class_indices = {
    'Brahmi (Bacopa monnieri)': 0,
    'Kallurukki (Aerva lanata)': 1,
    'Karisalankanni (Eclipta prostrata)': 2,
    'Keezhanelli (Phyllanthus niruri)': 3,
    'Kuthiraiveli (Cleome gynandra)': 4,
    'Mudakathan (Cardiospermum halicacabum)': 5,
    'Nilavembu / Siriyanangai (Andrographis paniculata)': 6,
    'Pirandai (Cissus quadrangularis)': 7,
    'Sirukurinjan (Gymnema sylvestre)': 8,
    'Vallarai (Centella asiatica)': 9
}
class_names = {v: k for k, v in class_indices.items()}

# ===============================
# MEDICINAL & AYURVEDIC USES (SEPARATE)
# ===============================
medicinal_uses = {
    0: "• Improves memory\n• Reduces anxiety\n• Neuroprotective",
    1: "• Kidney stone treatment\n• Urinary infections\n• Diuretic",
    2: "• Liver protection\n• Promotes hair growth",
    3: "• Hepatitis support\n• Kidney stone prevention",
    4: "• Joint pain relief\n• Improves digestion",
    5: "• Arthritis relief\n• Muscle stiffness reduction",
    6: "• Fever control\n• Boosts immunity",
    7: "• Bone fracture healing\n• Osteoporosis support",
    8: "• Diabetes control\n• Reduces sugar absorption",
    9: "• Memory enhancement\n• Wound healing"
}

ayurvedic_uses = {
    0: "• Medhya Rasayana\n• Epilepsy & insomnia\n• Balances Vata & Kapha",
    1: "• Mutrakrichra\n• Kidney detox\n• Balances Pitta & Kapha",
    2: "• Bhringraj\n• Liver disorders\n• Balances Vata & Kapha",
    3: "• Kamala (Jaundice)\n• Liver detox\n• Balances Pitta",
    4: "• Arthritis treatment\n• Digestive stimulant\n• Balances Vata",
    5: "• Vata disorders\n• Herbal oil therapy",
    6: "• Jwara (Fever)\n• Blood purifier\n• Balances Pitta & Kapha",
    7: "• Asthisamhara\n• Bone strengthening\n• Balances Vata",
    8: "• Madhumeha\n• Kapha balance",
    9: "• Medhya Rasayana\n• Skin diseases\n• Balances Vata & Pitta"
}

# ===============================
# IMAGE SIZE
# ===============================
IMG_HEIGHT = model.input_shape[1]
IMG_WIDTH = model.input_shape[2]

# ===============================
# PREPROCESS IMAGE (ALL FORMATS)
# ===============================
def preprocess_image(path):
    img = Image.open(path)

    if getattr(img, "is_animated", False):
        img = next(ImageSequence.Iterator(img))

    img = img.convert("RGB")
    img = img.resize((IMG_WIDTH, IMG_HEIGHT))

    img_array = np.array(img).astype("float32") / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    return img_array

# ===============================
# FILE SELECT & PREDICT
# ===============================
def choose_file():
    file_path = filedialog.askopenfilename(
        filetypes=[("All Image Files",
                    "*.jpg *.jpeg *.png *.bmp *.tiff *.tif *.webp *.jfif * .gif")]
    )

    if not file_path:
        return

    # Display image
    display_img = Image.open(file_path)
    display_img.thumbnail((260, 260))
    tk_img = ImageTk.PhotoImage(display_img)
    img_label.config(image=tk_img)
    img_label.image = tk_img

    try:
        img_array = preprocess_image(file_path)
        preds = model.predict(img_array, verbose=0)[0]

        idx = np.argmax(preds)
        confidence = preds[idx] * 100

        result_label.config(
            text=f"Prediction: {class_names[idx]}\nConfidence: {confidence:.2f}%",
            fg="green"
        )

        medicinal_text.delete(1.0, END)
        medicinal_text.insert(END, medicinal_uses[idx])

        ayurvedic_text.delete(1.0, END)
        ayurvedic_text.insert(END, ayurvedic_uses[idx])

    except Exception as e:
        result_label.config(text=f"Error: {e}", fg="red")

# ===============================
# TKINTER GUI
# ===============================
root = Tk()
root.title("Medicinal Plant Identification System")
root.geometry("600x760")
root.configure(bg="#eef5ff")

Label(
    root,
    text="Medicinal Plant Identification",
    font=("Arial", 18, "bold"),
    bg="#eef5ff"
).pack(pady=12)

Button(
    root,
    text="Choose Image",
    command=choose_file,
    font=("Arial", 13),
    bg="#2e7d32",
    fg="white",
    padx=20,
    pady=6
).pack(pady=8)

img_label = Label(root, bg="#eef5ff")
img_label.pack(pady=10)

result_label = Label(
    root,
    text="",
    font=("Arial", 13),
    bg="#eef5ff"
)
result_label.pack(pady=8)

# -------------------------------
# MEDICINAL USES
# -------------------------------
Label(
    root,
    text="Clinical / Medicinal Uses",
    font=("Arial", 14, "bold"),
    bg="#eef5ff"
).pack(pady=5)

medicinal_text = Text(root, height=6, width=65, font=("Arial", 11))
medicinal_text.pack(pady=5)

# -------------------------------
# AYURVEDIC USES
# -------------------------------
Label(
    root,
    text="Ayurvedic Uses",
    font=("Arial", 14, "bold"),
    bg="#eef5ff"
).pack(pady=5)

ayurvedic_text = Text(root, height=6, width=65, font=("Arial", 11))
ayurvedic_text.pack(pady=5)

root.mainloop()
